# Earth Data Hub: quickstart

Stream ERA5 data from Earth Data Hub. Same underlying data as the CDS
entries in this repo, but accessed as a cloud-optimised Zarr store rather
than downloaded as a file. No queue, no whole-file download.

See [`docs/earth-data-hub/README.md`](../docs/earth-data-hub/README.md) for
the full reference.

## Before you run this

1. Register at https://platform.destine.eu/
2. Generate a Personal Access Token in your EDH account settings.
3. Add it to `~/.netrc` as:
   ```
   machine data.earthdatahub.destine.eu
     login your-destine-username
     password YOUR_TOKEN
   ```
4. `pip install xarray zarr fsspec aiohttp netcdf4`

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
EDH_DATASET_URL = (
    "https://data.earthdatahub.destine.eu/era5/"
    "reanalysis-era5-single-levels-v0.zarr"
)
VARIABLE = "2m_temperature"
TIME_START = "2023-07-01T00:00"
TIME_END = "2023-07-01T23:00"
LAT_NORTH, LAT_SOUTH = 55, 49
LON_WEST, LON_EAST = -8, 2
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print(f"Python       {sys.version.split()[0]}")
for pkg in ["xarray", "zarr", "fsspec", "aiohttp", "matplotlib", "cartopy"]:
    try:
        print(f"{pkg:12} {version(pkg)}")
    except Exception:
        print(f"{pkg:12} (not installed)")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from common.credentials import check_edh_token
check_edh_token()
print("\nEDH token found.")

## Open the Zarr store lazily

This does not download the dataset. It connects to the remote Zarr store
and reads only the coordinate metadata, enough to describe the shape and
dimensions. `chunks={}` defers actual data reads until a `.load()` or
reduction is called.

The `storage_options` with `trust_env=True` tells the underlying
aiohttp session to read credentials from `~/.netrc`.

In [ ]:
ds = xr.open_dataset(
    EDH_DATASET_URL,
    engine="zarr",
    chunks={},
    storage_options={"client_kwargs": {"trust_env": True}},
)
print(ds)

## Subset in space and time

Still lazy. `.sel()` defines which byte-ranges will be fetched when we
finally call `.load()`.

ERA5 longitude is typically 0 to 360 rather than -180 to 180. If your
bounding box spans the Greenwich meridian or uses negative longitudes,
you may need to shift: `ds = ds.assign_coords(longitude=(ds.longitude + 180) % 360 - 180).sortby("longitude")`
before slicing. The example below assumes the dataset was already
re-indexed (EDH hosts variants that are).

In [ ]:
subset = ds[VARIABLE].sel(
    time=slice(TIME_START, TIME_END),
    latitude=slice(LAT_NORTH, LAT_SOUTH),
    longitude=slice(LON_WEST, LON_EAST),
)
print("Lazy subset shape:", subset.shape)
print("Size in memory if loaded:", subset.nbytes / 1e6, "MB")

## Materialise and plot

Now we actually fetch bytes. For this small UK/one-day subset the transfer
is fast. `.mean("time")` reduces to a 2D field for a simple map.

In [ ]:
daily_mean = (subset.mean("time") - 273.15).load()

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())
daily_mean.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    cmap="RdBu_r",
    cbar_kwargs={"label": "Daily mean 2 m temperature (C)"},
)
ax.coastlines(resolution="50m", linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor="gray")
gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5)
gl.top_labels = False
gl.right_labels = False
ax.set_title(f"ERA5 via Earth Data Hub: UK daily-mean 2 m temperature, 2023-07-01")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/earth-data-hub/README.md`](../docs/earth-data-hub/README.md)
- **Long point time series**: the streaming model shines here. Try
  `ds["2m_temperature"].sel(latitude=51.5, longitude=-0.1, method="nearest")`
  over a 10-year time range; you only pay for the pixels at that point.
- **Other datasets**: browse https://earthdatahub.destine.eu/catalogue
  for ERA5-Land, CMIP6 models, Climate DT output, and Copernicus DEM.
- **Quota awareness**: 500,000 requests per user per month. Large lazy
  slicing of small-chunked datasets can burn through this; subset
  aggressively before `.load()`.